# Quantification — Foram adaptation

# [FORAM ADAPTATION] This notebook is a copy of Behnaz's quantification.ipynb,
# modified minimally for the MOM foram dataset. All changes are marked with
# [FORAM ADAPTATION] comments, with the original line preserved above each change.
#
# Input data: data/final_foram_state/clustered_volume/*.npy
# 3-D uint8 arrays: 0=background, 1=shell, 2+=chamber clusters (descending pore count)

In [1]:
import os
import re
import time
import glob
from pathlib import Path

import numpy as np
import pandas as pd

import porespy as ps
import scipy.ndimage as ndi  # Import for connected component labeling 

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from tqdm.notebook import tqdm
from datetime import datetime

# start_time = time.time()
# print("--- %s seconds ---" % (time.time() - start_time))
# from tqdm.notebook import tqdm


In [2]:
import skimage
print(skimage.__version__)

0.24.0


In [ ]:
# Function to compute local thickness and measurements
def compute_metrics(data):
    """
    Compute local thickness and volume-related measurements for a given class.
    Filters out zero values from local thickness calculations.
    """
    # Create binary mask
    binary_data = (data > 0)  # Ensure only nonzero values are considered

    # [FORAM ADAPTATION] Guard against empty masks (can occur for very small chambers)
    if not binary_data.any():
        return {
            "Total_Volume": 0, "Num_Pores": 0,
            "Max_Pore_Volume": 0, "Min_Pore_Volume": 0,
            "Std_Pore_Volume": 0, "Mean_Pore_Volume": 0,
            "Max_LT": 0, "Min_LT": 0, "Std_LT": 0, "Mean_LT": 0,
            "LT_Distribution": np.array([]),
        }

    # Compute local thickness using porespy
    # [FORAM ADAPTATION] mode='dt' -> method='dt'  (API change in installed porespy version)
    # Original: local_thickness = ps.filters.local_thickness(binary_data, mode='dt')
    local_thickness = ps.filters.local_thickness(binary_data, method='dt')

    # Remove zeros from local thickness array
    lt_nonzero = local_thickness[local_thickness > 0]

    # Compute volume metrics
    total_volume = np.sum(binary_data)

    # Label connected components (pores)
    labeled_pores, num_pores = ndi.label(binary_data)

    # Compute pore volumes, filtering out zero regions
    pore_volumes = [np.sum(labeled_pores == i) for i in range(1, num_pores + 1)]

    # Ensure we are not computing statistics on empty arrays
    metrics = {
        "Total_Volume": total_volume,
        "Num_Pores": num_pores,
        "Max_Pore_Volume": max(pore_volumes) if pore_volumes else 0,
        "Min_Pore_Volume": min(pore_volumes) if pore_volumes else 0,
        "Std_Pore_Volume": np.std(pore_volumes) if pore_volumes else 0,
        "Mean_Pore_Volume": np.mean(pore_volumes) if pore_volumes else 0,
        "Max_LT": np.max(lt_nonzero) if lt_nonzero.size > 0 else 0,
        "Min_LT": np.min(lt_nonzero) if lt_nonzero.size > 0 else 0,
        "Std_LT": np.std(lt_nonzero) if lt_nonzero.size > 0 else 0,
        "Mean_LT": np.mean(lt_nonzero) if lt_nonzero.size > 0 else 0,
        "LT_Distribution": lt_nonzero  # Only nonzero local thickness values
    }

    return metrics

# Our data

In [ ]:
# [FORAM ADAPTATION] Updated paths for foram project structure
# Original:
# DATA_FOLDER = "../Prediction/February"
# OUTPUT_RESULTS = "../Prediction/February/results"
# OUTPUT_EXCEL = "../Prediction/February/results/quantification_results.xlsx"
from pathlib import Path
PROJECT_ROOT = Path('/home/hanqingwu/lu2026-17-19/Porosity/porosity_segmentation')  # Adjust as needed to point to the root of your project
DATA_FOLDER = str(PROJECT_ROOT / 'data/final_foram_state/clustered_volume')
OUTPUT_RESULTS = str(PROJECT_ROOT / 'data/analysis/quantification')
OUTPUT_EXCEL = str(PROJECT_ROOT / 'data/analysis/quantification/quantification_results.csv')

# Create output directory if not exists
os.makedirs(OUTPUT_RESULTS, exist_ok=True)

In [ ]:
# Initialize an empty list to store results
results = []

file_list = os.listdir(DATA_FOLDER)

# Process each .npy file in the folder
for file_name in tqdm(file_list, desc="Processing Files", ascii=True, dynamic_ncols=False):
    print(file_name)

    if file_name.endswith(".npy"):
        file_path = os.path.join(DATA_FOLDER, file_name)

        # Start time measurement
        start_time = time.time()

        # Load the .npy file
        data = np.load(file_path)

        # [FORAM ADAPTATION] Our data is 3-D integer-labelled (not 4-D (x,y,z,3))
        # Original: if data.ndim != 4 or data.shape[-1] != 3:
        if data.ndim != 3:
            print(f"Skipping {file_name} due to unexpected shape {data.shape}")
            continue

        # [FORAM ADAPTATION] Extract masks from integer labels instead of channel slices
        # Original: pores_metrics = compute_metrics(data[...,0])
        #           shell_metrics = compute_metrics(1-data[...,2])
        pores_metrics = compute_metrics((data >= 2).astype(np.uint8))  # all pore voxels
        shell_metrics = compute_metrics((data == 1).astype(np.uint8))  # shell only

        # Save LT distributions
        lt_distributions = {
            'pores': pores_metrics["LT_Distribution"],
            'shell': shell_metrics["LT_Distribution"]
        }
        lt_save_path = os.path.join(OUTPUT_RESULTS, f"{file_name}_LT.npy")
        np.save(lt_save_path, lt_distributions)

        # Compute elapsed time
        elapsed_time = time.time() - start_time

        # Store the results
        results.append({
            "File_Name": file_name,
            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Processing_Time_Seconds": round(elapsed_time, 2),
            "Pore_Total_Volume": pores_metrics["Total_Volume"],
            "Pore_Num": pores_metrics["Num_Pores"],
            "Pore_Max_Volume": pores_metrics["Max_Pore_Volume"],
            "Pore_Min_Volume": pores_metrics["Min_Pore_Volume"],
            "Pore_Std_Volume": pores_metrics["Std_Pore_Volume"],
            "Pore_Mean_Volume": pores_metrics["Mean_Pore_Volume"],
            "Pore_Max_LT": pores_metrics["Max_LT"],
            "Pore_Min_LT": pores_metrics["Min_LT"],
            "Pore_Std_LT": pores_metrics["Std_LT"],
            "Pore_Mean_LT": pores_metrics["Mean_LT"],
            "Shell_Total_Volume": shell_metrics["Total_Volume"],
            "Shell_Max_LT": shell_metrics["Max_LT"],
            "Shell_Min_LT": shell_metrics["Min_LT"],
            "Shell_Std_LT": shell_metrics["Std_LT"],
            "Shell_Mean_LT": shell_metrics["Mean_LT"],
        })

# Convert results into a DataFrame and save
# [FORAM ADAPTATION] Save as CSV instead of Excel (no openpyxl needed on cluster)
# Original: df.to_excel(OUTPUT_EXCEL, index=False)
df = pd.DataFrame(results)
df.to_csv(OUTPUT_EXCEL, index=False)

print(f"Quantification complete. Results saved in '{OUTPUT_EXCEL}'. LT arrays saved in '{OUTPUT_RESULTS}'.")

# Statistics on chambers

In [ ]:
# [FORAM ADAPTATION] Updated paths for foram project (same DATA_FOLDER/OUTPUT_EXCEL as above)
# Original:
# DATA_FOLDER = "../Prediction/ClusterInfo"
# OUTPUT_EXCEL = "../Prediction/ClusterInfo/results/quantification_results_cluster.xlsx"
# OUTPUT_CLUSTER_EXCEL = "../Prediction/ClusterInfo/results_class_mapped/quantification_results_cluster.xlsx"
# (DATA_FOLDER and OUTPUT_RESULTS already set in the paths cell above)

In [ ]:
# read the 5 first cluster index
cluster_index_EXCEL = "../Prediction/top5_num_pores.xlsx"

# Load the Excel file
df = pd.read_excel(cluster_index_EXCEL)  # Replace with your actual file name

# Define the cluster ID columns in order
cluster_cols = [
    'Cluster ID New 1',
    'Cluster ID New 2',
    'Cluster ID New 3',
    'Cluster ID New 4',
    'Cluster ID New 5'
]

output_rows = []

# Process each row
for idx, row in df.iterrows():
    file_name = row['File_Name']

    # Stop if File_Name is missing — likely end of real data
    if pd.isna(file_name):
        print(f"Stopping at row {idx}: empty File_Name.")
        break

    # Extract cluster values safely
    cluster_values = [int(row[col]) for col in cluster_cols]

    # Build row
    output_row = {'File_Name': file_name}
    for i, cluster_id in enumerate(cluster_values):
        output_row[f'Position_{i+1}'] = cluster_id

    output_rows.append(output_row)

# Create and save the output
output_df = pd.DataFrame(output_rows)
output_df.to_excel("../Prediction/ClusterInfo/cluster_position_mapping.xlsx", index=False)

print("Cluster positions saved to 'cluster_position_mapping.xlsx'")

In [23]:
npy_files = []
for root, dirs, files in os.walk(DATA_FOLDER):
    for file in files:
        if file.endswith('.npy'):
            npy_files.append(os.path.join(root, file))

In [ ]:
# Process and plot each file
for file_path in tqdm(npy_files, desc="Plotting middle Z-slices", ascii=True, dynamic_ncols=False):
    data = np.load(file_path)

    # Check data dimensions (assumes data has at least 3 dimensions)
    if data.ndim < 3:
        print(f"Skipping {file_path}, data shape {data.shape} is insufficient.")
        continue

    # Compute the middle Z slice
    z_middle = data.shape[2] // 2
    slice_data = data[:, :, z_middle]

    # Plotting
    plt.figure(figsize=(6, 6))
    plt.imshow(slice_data, cmap='gray')  # or use another colormap as needed
    plt.title(f"Middle Z-Slice\n{os.path.basename(file_path)}")
    plt.axis('off')  # Hide axis ticks for clarity
    
    # Save the plot next to the original .npy file
    output_path = file_path.replace('.npy', '_middle_z.png')
    plt.savefig(output_path, bbox_inches='tight')
    plt.close()

print("All middle Z-slices plotted and saved successfully.")

In [ ]:
results = []

OUTPUT_RESULTS = str(PROJECT_ROOT / 'data/analysis/quantification')
os.makedirs(OUTPUT_RESULTS, exist_ok=True)

# [FORAM ADAPTATION] Collect npy_files from DATA_FOLDER (set above)
# Original used a pre-built npy_files list from os.walk(DATA_FOLDER)
import glob as _glob
npy_files = sorted(_glob.glob(os.path.join(DATA_FOLDER, '*.npy')))

# Process each file
for file_path in tqdm(npy_files, desc="Processing Files", ascii=True, dynamic_ncols=False):
    file_name = os.path.basename(file_path)
    print(f"Processing: {file_name}")
    data = np.load(file_path)

    if data.ndim != 3:
        print(f"Skipping {file_name}, expected shape (x,y,z), found {data.shape}")
        continue

    # Identify unique classes/clusters in the data
    class_labels = np.unique(data)
    # [FORAM ADAPTATION] Skip background (0) AND shell (1); chambers start at 2
    # Original: class_labels = class_labels[class_labels > 0]
    class_labels = class_labels[class_labels >= 2]

    for class_label in class_labels:
        start_time = time.time()

        # Extract binary mask for current class
        class_data = (data == class_label).astype(np.uint8)

        # Compute metrics for current class
        pores_metrics = compute_metrics(class_data)

        # Save LT distribution to .npy file
        lt_values = pores_metrics["LT_Distribution"]
        lt_values = lt_values[lt_values > 0]
        lt_filename = os.path.join(OUTPUT_RESULTS, f"{file_name}_class_{class_label}_LT.npy")
        np.save(lt_filename, lt_values)

        elapsed_time = time.time() - start_time

        # Save results clearly per file and class
        results.append({
            "File_Name": f"{file_name}_class_{class_label}",
            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "Processing_Time_Seconds": round(elapsed_time, 2),
            "Pore_Total_Volume": pores_metrics["Total_Volume"],
            "Pore_Num": pores_metrics["Num_Pores"],
            "Pore_Max_Volume": pores_metrics["Max_Pore_Volume"],
            "Pore_Min_Volume": pores_metrics["Min_Pore_Volume"],
            "Pore_Std_Volume": pores_metrics["Std_Pore_Volume"],
            "Pore_Mean_Volume": pores_metrics["Mean_Pore_Volume"],
            "Pore_Max_LT": pores_metrics["Max_LT"],
            "Pore_Min_LT": pores_metrics["Min_LT"],
            "Pore_Std_LT": pores_metrics["Std_LT"],
            "Pore_Mean_LT": pores_metrics["Mean_LT"],
        })

        # Plot Ridge plot for Local Thickness distribution per class
        if lt_values.size > 0:
            plt.figure(figsize=(8, 6))
            sns.kdeplot(lt_values, fill=True, bw_adjust=0.5)
            plt.title(f"Local Thickness Distribution\n{file_name} - Class {class_label}")
            plt.xlabel("Local Thickness")
            plt.ylabel("Density")
            plt.grid()
            plot_filename = f"{file_name}_class_{class_label}_LT.png"
            plt.savefig(os.path.join(OUTPUT_RESULTS, plot_filename))
            plt.close()

# [FORAM ADAPTATION] Save as CSV instead of Excel
# Original: df.to_excel(OUTPUT_EXCEL, index=False)
df = pd.DataFrame(results)
df.to_csv(OUTPUT_EXCEL, index=False)

print(f"Quantification complete. Results saved in '{OUTPUT_EXCEL}'.")
print(f"Ridge plots and LT distributions saved in '{OUTPUT_RESULTS}'.")

# Visualize the results

In [29]:
# for each sample show distribution of the LT of pores and each chambers

# Read the Excel file (update 'your_file.xlsx' with your file name)
df = pd.read_excel("../Prediction/ClusterInfo/results/quantification_results_cluster.xlsx")

In [ ]:
# Load the cluster position mapping
mapping_df = pd.read_excel("../Prediction/ClusterInfo/results/cluster_position_mapping.xlsx")

# Extract base file name (without class) to match
df['Base_File'] = df['File_Name'].apply(lambda x: str(x).split('_class_')[0] if '_class_' in str(x) else str(x))

# Prepare output
output_rows = []

# Process each base file in the mapping
for _, map_row in mapping_df.iterrows():
    base_file = str(map_row['File_Name'])  # e.g. "DV8_sp6_1993_clustered.npy"
    cluster_ids = [str(map_row[f'Position_{i}']) for i in range(1, 6)]
    
    # For each class (1 to 5), find the row in original data with that cluster ID
    for new_class_index, original_cluster_id in enumerate(cluster_ids, start=1):
        # Find matching row in original data
        class_row = df[
            (df['Base_File'].str.contains(base_file.replace('_clustered.npy', ''))) &
            (df['File_Name'].str.endswith(f'_class_{original_cluster_id}'))
        ]
        
        if not class_row.empty:
            row_copy = class_row.copy()
            # Add original class number column
            row_copy['Original_Class'] = original_cluster_id
            # Update filename with new class number
            row_copy['File_Name'] = row_copy['File_Name'].str.replace(
                f'_class_{original_cluster_id}',
                f'_class_{new_class_index}'
            )
            output_rows.append(row_copy)

# Concatenate all into final DataFrame if we have rows
if output_rows:
    final_df = pd.concat(output_rows, ignore_index=True)
    
    # Drop helper column
    final_df.drop(columns=['Base_File'], inplace=True)
    
    # Save to Excel
    final_df.to_excel("../Prediction/ClusterInfo/remapped_filtered_classes.xlsx", index=False)
    print("Remapped and filtered data saved to 'remapped_filtered_classes.xlsx'")
else:
    print("No matching data found to process")

In [ ]:
# Read the mapping file
mapping_df = pd.read_excel("../Prediction/ClusterInfo/results/cluster_position_mapping.xlsx")

# Create the output directory
output_dir = "../Prediction/ClusterInfo/results_class_mapped"
os.makedirs(output_dir, exist_ok=True)

# Get list of all .npy files in the source directory
source_dir = "../Prediction/ClusterInfo/results"
npy_files = [f for f in os.listdir(source_dir) if f.endswith('.npy')]

def find_new_position(original_class, row):
    # Check which position (1-5) contains the original class
    for pos in range(1, 6):
        if row[f'Position_{pos}'] == original_class:
            return pos
    return None

# Process each file
for file in npy_files:
    try:
        # Extract the current class number from the filename
        # Example: 116_MOM_E_clavatum_31_02_clustered.npy_class_9_LT.npy
        current_class = int(file.split('class_')[1].split('_')[0])
        
        # Get the base filename without class info
        base_filename = file.split('clustered.npy')[0] + 'clustered.npy'
        
        # Find the corresponding row in mapping_df
        mapping_row = mapping_df[mapping_df['File_Name'] == base_filename].iloc[0]
        
        # Find which position contains the current class
        new_class = find_new_position(current_class, mapping_row)
        
        if new_class is not None:
            # Create new filename by replacing old class with new class
            new_filename = file.replace(f'class_{current_class}', f'class_{new_class}')
            
            # Load the data
            data = np.load(os.path.join(source_dir, file))
            
            # Save with new filename in the new directory
            np.save(os.path.join(output_dir, new_filename), data)
            print(f"Processed: {file} -> {new_filename}")
        
    except Exception as e:
        print(f"Error processing file {file}: {str(e)}")

print("Processing complete. Files have been saved in the results_class_mapped directory.") 

In [23]:
def plot_thickness_comparison(shell_thickness, pore_thickness, chamber_pore_thickness):
    """
    Create a clean visualization comparing thickness distributions.
    """

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Local Thickness Analysis', fontsize=18, fontweight='bold')

    # -- SHELL THICKNESS --
    sns.histplot(shell_thickness, bins='fd', stat='density', 
                 color='skyblue', edgecolor='black', ax=axes[0])
    sns.kdeplot(shell_thickness, color='red', linewidth=2, bw_adjust=0.5, cut=0, ax=axes[0])
    axes[0].set_title('Shell Thickness Distribution', fontsize=14)
    axes[0].set_xlabel('Thickness (voxels)')
    axes[0].set_ylabel('Density')
    axes[0].grid(alpha=0.3)

    # -- PORE THICKNESS --
    sns.histplot(pore_thickness, bins='fd', stat='density', 
                 color='skyblue', edgecolor='black', ax=axes[1])
    sns.kdeplot(pore_thickness, color='red', linewidth=2, bw_adjust=0.6, cut=0, ax=axes[1])
    axes[1].set_title('Pore Thickness Distribution', fontsize=14)
    axes[1].set_xlabel('Thickness (voxels)')
    axes[1].set_ylabel('Density')
    axes[1].grid(alpha=0.3)

    # -- CHAMBER VIOLIN PLOT --
    chamber_data = []
    chamber_labels = []
    for chamber, thickness in chamber_pore_thickness.items():
        chamber_data.append(thickness)
        chamber_labels.extend([f'Chamber {chamber}'] * len(thickness))

    flat_data = np.concatenate(chamber_data)
    df = pd.DataFrame({
        'Thickness': flat_data,
        'Chamber': chamber_labels
    })

    sns.violinplot(data=df, x='Chamber', y='Thickness', ax=axes[2],
                   bw_method=0.3, inner='box', cut=0)
    axes[2].set_title('Chamber Pore Thickness Distribution', fontsize=14)
    axes[2].set_xlabel('Chamber')
    axes[2].set_ylabel('Thickness')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].grid(alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
def main():
    # Define the base directories where your data is stored
    cluster_dir = Path('../Prediction/ClusterInfo/results_class_mapped')
    feb_dir = Path('../Prediction/February/results')
    
    # Find all LT files in the February results
    lt_files = list(feb_dir.glob('*_LT.npy'))
    print(f"\nFound {len(lt_files)} LT files")
    
    # Process only the first two samples
    for lt_file in lt_files[0:2]:
        sample_name = lt_file.stem.replace('_LT', '')  # Remove _LT from filename
        print(f"\nProcessing sample: {sample_name}")
        
        try:
            # Load the dictionary containing both shell and pore thickness data
            lt_data = np.load(lt_file, allow_pickle=True).item()
            shell_thickness = lt_data['shell']
            pore_thickness = lt_data['pores']
            
            # Check if thickness data is empty/invalid
            if shell_thickness.size == 0 or pore_thickness.size == 0:
                print(f"Warning: Empty thickness data for {sample_name}, skipping...")
                continue
                
            # Extract base name without the 3d_volume_uint8_UNet.npy suffix
            base_name = '_'.join(sample_name.split('_')[:6])
            
            # Find all chamber pore thickness files
            chamber_files = glob.glob(str(cluster_dir / f"{base_name}_clustered.npy_class_*_LT.npy"))
            
            # Sort chamber files by chamber number
            chamber_files = sorted(chamber_files, 
                                 key=lambda x: int(x.split('class_')[-1].split('_')[0]))
            
            chamber_pore_thickness = {}
            for chamber_file in chamber_files:
                # Extract chamber number from filename
                chamber_num = chamber_file.split('class_')[-1].split('_')[0]
                chamber_data = np.load(chamber_file)
                
                # Check if chamber data is valid
                if chamber_data.size > 0:
                    chamber_pore_thickness[chamber_num] = chamber_data
                else:
                    print(f"Warning: Empty data for chamber {chamber_num}, skipping...")
            
            # Only proceed with visualization if we have valid chamber data
            if chamber_pore_thickness:
                # Create visualizations
                print("\nCreating visualizations...")
                
                # Combined comparison
                plot_thickness_comparison(shell_thickness, pore_thickness, chamber_pore_thickness)
                
            
        except Exception as e:
            print(f"Error processing {sample_name}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print("\nAll processing complete.")

if __name__ == "__main__":
    main() 

In [27]:
# Function to generate n distinct colors using the HSV color space.
def generate_distinct_colors(n):
    colors = []
    for i in range(n):
        hue = i / n
        # Here, we fix saturation and value to constants (0.8) for vivid colors.
        colors.append(mcolors.hsv_to_rgb([hue, 0.8, 0.8]))
    return colors

In [44]:
# Read the Excel file (update 'your_file.xlsx' with your file name)
df = pd.read_excel("../Prediction/ClusterInfo/remapped_filtered_classes.xlsx")

In [ ]:

df['Group'] = df['File_Name']
unique_groups = df['Group'].unique()
n_groups = len(unique_groups)

# Generate a list of distinct colors
distinct_colors = generate_distinct_colors(n_groups)
group_to_color = {group: distinct_colors[i] for i, group in enumerate(unique_groups)}

# ----------------------------
# Scatter Plot: Pore_Total_Volume vs Pore_Num with distinct colors for each group
# ----------------------------
plt.figure(figsize=(12, 8))
for group, sub_df in df.groupby('Group'):
    plt.scatter(sub_df['Pore_Total_Volume']/1e5, sub_df['Pore_Num'],
                label=group,
                color=group_to_color[group],
                edgecolor='k',
                alpha=0.7)
plt.xlabel('Pore Total Volume')
plt.ylabel('Pore Number')
plt.title('Scatter Plot: Pore Total Volume vs Pore Number by Group')
# With many groups, the legend can be overwhelming. You might opt to display only a subset,
# or place it outside the plot.
plt.legend(fontsize='small', ncol=2, title='Group', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Split the File_Name into Dataset_Base and Class.
df[['Dataset_Base', 'Class']] = df['File_Name'].str.split('_class_', expand=True)
df['Class'] = df['Class'].astype(int)

# For each dataset base, create one scatter plot that shows all its classes
for dataset, dataset_df in df.groupby('Dataset_Base'):
    # Get unique classes in the current dataset.
    unique_classes = sorted(dataset_df['Class'].unique())
    n_classes = len(unique_classes)
    
    # Use the viridis colormap to generate distinct colors for each class.
    cmap = plt.get_cmap('viridis', n_classes)
    class_to_color = {cls: cmap(i) for i, cls in enumerate(unique_classes)}
    
    # Create figure and axes with space for legend on right
    fig, ax = plt.subplots(figsize=(5, 3))
    
    for cls, sub_df in dataset_df.groupby('Class'):
        ax.scatter(sub_df['Pore_Total_Volume'] / 1e5, sub_df['Pore_Num'],
                    label=f'Class {cls}',
                    color=class_to_color[cls],
                    edgecolor='k',
                    alpha=0.7)
    ax.set_xlabel('Pore Total Volume (scaled by 1e5)')
    ax.set_ylabel('Pore Number')
    ax.set_title(f'Scatter Plot for {dataset}')
    
    # Place legend outside plot on the right
    plt.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Adjust layout to prevent legend cutoff
    plt.tight_layout()
    plt.show()

In [ ]:
# Split the File_Name into Dataset_Base and Class
df[['Dataset_Base', 'Class']] = df['File_Name'].str.split('_class_', expand=True)
df['Class'] = df['Class'].astype(int)

# Loop over each unique dataset
for dataset in df['Dataset_Base'].unique():
    print(dataset)
    dataset_df = df[df['Dataset_Base'] == dataset]
    classes = sorted(dataset_df['Class'].unique())
    x_positions = np.arange(len(classes))

    
    # Prepare lists to store statistics for each class
    vol_min, vol_mean, vol_max = [], [], []
    lt_min, lt_mean, lt_max = [], [], []
    
    # Compute statistics for each class
    for cls in classes:
        class_group = dataset_df[dataset_df['Class'] == cls]
        
        # For Volume per Pore
        vol_min.append(class_group['Pore_Min_Volume'])
        vol_mean.append(class_group['Pore_Mean_Volume'])
        vol_max.append(class_group['Pore_Max_Volume'])
        
        # For Pore LT
        lt_min.append(class_group['Pore_Min_LT'])
        lt_mean.append(class_group['Pore_Mean_LT'])
        lt_max.append(class_group['Pore_Max_LT'])

    # Create a figure with two subplots side by side
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Left subplot: Volume per Pore statistics
    axes[0].vlines(x_positions, vol_min, vol_max, color='blue', lw=2)
    axes[0].scatter(x_positions, vol_mean, color='red', s=100, zorder=3)
    axes[0].set_xticks(x_positions)
    axes[0].set_xticklabels([str(cls) for cls in classes])
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Pore Volume')
    axes[0].set_title(f'{dataset} - Pore Volume Statistics')
    axes[0].grid(True, axis='y', linestyle='--', alpha=0.7)
    
    # Right subplot: Pore LT statistics
    axes[1].vlines(x_positions, lt_min, lt_max, color='blue', lw=2)
    axes[1].scatter(x_positions, lt_mean, color='red', s=100, zorder=3)
    axes[1].set_xticks(x_positions)
    axes[1].set_xticklabels([str(cls) for cls in classes])
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Pore LT')
    axes[1].set_title(f'{dataset} - Pore LT Statistics')
    axes[1].grid(True, axis='y', linestyle='--', alpha=0.7)
    
    fig.suptitle(f'Statistics for {dataset}', fontsize=18)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

## Test the difference between the chambers

#### Prepare the data

In [10]:
def process_sample_clusters(data_folder, mapping_file, output_dir):
    """
    Process sample data and create separate .npy files for each mapped cluster.
    
    Parameters:
    -----------
    data_folder : str
        Path to the folder containing sample data
    mapping_file : str
        Path to the Excel file containing cluster mapping information
    output_dir : str
        Path to save the output files
    """
    # Read the mapping file
    mapping_df = pd.read_excel(mapping_file)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all .npy files from the data folder and its subfolders
    npy_files = []
    for root, dirs, files in os.walk(data_folder):
        for file in files:
            if file.endswith('.npy'):
                npy_files.append(os.path.join(root, file))
    
    def create_mapped_arrays(data, mapping_row):
        """Create 5 arrays for each mapped position"""
        mapped_arrays = {}
        
        # Initialize arrays for positions 1-5
        for pos in range(1, 6):
            mapped_arrays[pos] = np.zeros_like(data)
        
        # For each unique value in the data
        for original_class in np.unique(data):
            if original_class == 0:  # Skip background
                continue
                
            # Find which position contains this class
            for pos in range(1, 6):
                if mapping_row[f'Position_{pos}'] == original_class:
                    mapped_arrays[pos][data == original_class] = 1
                    break
        
        return mapped_arrays
    
     # Process each file
    for npy_file in npy_files:
        try:
            # Get the sample name from the folder structure
            sample_name = os.path.basename(os.path.dirname(npy_file))
            
            # Load the data
            data = np.load(npy_file)
            
            # Find the corresponding row in mapping_df
            base_filename = os.path.basename(npy_file)
            # Remove "_refined" from filename for mapping lookup
            lookup_filename = base_filename.replace("_refined.npy", ".npy")
            
            # Print debug information
            print(f"\nProcessing file: {base_filename}")
            print(f"Looking up mapping for: {lookup_filename}")
            
            # Find the mapping row
            mapping_row = mapping_df[mapping_df['File_Name'] == lookup_filename]
            if mapping_row.empty:
                print(f"Warning: No mapping found for {lookup_filename}")
                continue
            mapping_row = mapping_row.iloc[0]
            
            # Create sample output directory
            sample_output_dir = os.path.join(output_dir, sample_name)
            os.makedirs(sample_output_dir, exist_ok=True)
            
            # Create mapped arrays
            mapped_arrays = create_mapped_arrays(data, mapping_row)
            
            # Save mapped arrays
            for pos, array in mapped_arrays.items():
                output_filename = f"{sample_name}_cluster_{pos}.npy"
                output_path = os.path.join(sample_output_dir, output_filename)
                np.save(output_path, array)
                
            print(f"Successfully processed: {sample_name} -> Created 5 cluster arrays")
            
        except Exception as e:
            print(f"Error processing file {npy_file}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print("\nProcessing complete. Files have been saved in the output directory.")


In [ ]:
# Set up paths
DATA_FOLDER = "../Prediction/ClusterInfo"
MAPPING_FILE = "../Prediction/ClusterInfo/cluster_position_mapping.xlsx"
OUTPUT_DIR = "../Prediction/ClusterInfo/mapped_clusters"

# Run the processing
process_sample_clusters(DATA_FOLDER, MAPPING_FILE, OUTPUT_DIR) 

Change the samples in ClusterInfo with RefinedClusterInfo:

In [11]:
# Set up paths
REFINED_DATA_FOLDER = "../Prediction/RefinedClusterInfo"
MAPPING_FILE = "../Prediction/ClusterInfo/cluster_position_mapping.xlsx"
OUTPUT_DIR = "../Prediction/RefinedClusterInfo/mapped_clusters"

# Get list of samples in refined folder
refined_samples = []
for root, dirs, files in os.walk(REFINED_DATA_FOLDER):
    for file in files:
        if file.endswith('.npy'):
            refined_samples.append(os.path.join(root, file))

if refined_samples:
    print(f"Found {len(refined_samples)} samples in refined folder:")
    for sample in refined_samples:
        print(f"  - {os.path.basename(sample)}")
    
    # Run the existing process_sample_clusters with the refined data folder
    process_sample_clusters(REFINED_DATA_FOLDER, MAPPING_FILE, OUTPUT_DIR)
else:
    print("No .npy files found in the refined folder.")

Found 27 samples in refined folder:
  - DV8-sp6-1993_clustered_refined.npy
  - DV5-sp8-2002_clustered_refined.npy
  - DV26-4_clustered_refined.npy
  - DV5-sp4-2002_clustered_refined.npy
  - 096_MOM_E_clavatum_3_03_clustered_refined.npy
  - 097_MOM_E_clavatum_3_04_clustered_refined.npy
  - 121_MOM_E_clavatum_31_07_clustered_refined.npy
  - 098_MOM_E_clavatum_3_05_clustered_refined.npy
  - 109_MOM_E_clavatum_12_01_clustered_refined.npy
  - 069_MOM_E_clavatum_210_08_clustered_refined.npy
  - DV24-05_clustered_refined.npy
  - DV8-sp2-1993_clustered_refined.npy
  - 087_MOM_E_clavatum_330_02_clustered_refined.npy
  - DV8-sp3-1993_clustered_refined.npy
  - 110_MOM_E_clavatum_12_02_clustered_refined.npy
  - 065_MOM_E_clavatum_210_04_clustered_refined.npy
  - 094_MOM_E_clavatum_3_01_clustered_refined.npy
  - 084_MOM_E_clavatum_300_07_clustered_refined.npy
  - 116_MOM_E_clavatum_31_02_clustered_refined.npy
  - DV4-sp1-2005_clustered_refined.npy
  - DV5-sp6-2002_clustered_refined.npy
  - 115_MOM_

In [13]:
def calculate_surface_area(binary_mask):
    """
    Calculate the surface area of a binary object using morphological operations.
    """
    # Create a structure for 6-connectivity
    struct = np.array([[[0,0,0],
                       [0,1,0],
                       [0,0,0]],
                      [[0,1,0],
                       [1,1,1],
                       [0,1,0]],
                      [[0,0,0],
                       [0,1,0],
                       [0,0,0]]], dtype=bool)
    
    # Erode the binary mask
    eroded = ndi.binary_erosion(binary_mask, structure=struct)
    
    # Surface voxels are those in the original mask that are not in the eroded mask
    surface = binary_mask & ~eroded
    
    # Count surface voxels
    return np.sum(surface)

def calculate_cylindricity(pore_mask):
    """
    Calculate how cylindrical a pore is by comparing its shape to an ideal cylinder.
    Returns a value between 0 and 1, where 1 indicates a perfect cylinder.
    """
    # Get the coordinates of all points in the pore
    coords = np.array(np.where(pore_mask)).T
    
    if len(coords) < 3:  # Need at least 3 points to define a cylinder
        return 0
    
    # Calculate the principal components to find the main axis
    mean = np.mean(coords, axis=0)
    centered_coords = coords - mean
    cov = np.cov(centered_coords.T)
    eigenvals, eigenvecs = np.linalg.eigh(cov)
    
    # Sort eigenvalues and eigenvectors in descending order
    idx = eigenvals.argsort()[::-1]
    eigenvals = eigenvals[idx]
    eigenvecs = eigenvecs[:, idx]
    
    # The largest eigenvector represents the main axis of the cylinder
    main_axis = eigenvecs[:, 0]
    
    # Project points onto plane perpendicular to main axis
    proj_matrix = np.eye(3) - np.outer(main_axis, main_axis)
    projected = np.dot(centered_coords, proj_matrix)
    
    # Calculate distances from points to main axis
    radial_distances = np.linalg.norm(projected, axis=1)
    
    # Calculate mean radius and height
    mean_radius = np.mean(radial_distances)
    height = np.max(np.dot(centered_coords, main_axis))
    
    # Calculate volume of ideal cylinder with same height and mean radius
    ideal_volume = np.pi * mean_radius**2 * height
    actual_volume = np.sum(pore_mask)
    
    # Calculate cylindricity as ratio of volumes, normalized to be between 0 and 1
    cylindricity = min(actual_volume / ideal_volume, ideal_volume / actual_volume) if ideal_volume > 0 else 0
    
    return cylindricity

def calculate_sphericity(volume, surface_area):
    """
    Calculate sphericity of a pore using volume and surface area.
    Sphericity = (36π * V^2)^(1/3) / S
    where V is volume and S is surface area.
    Returns a value between 0 and 1, where 1 indicates a perfect sphere.
    The ratio is normalized by taking the minimum of the calculated value and 1.
    """
    if surface_area == 0:
        return 0
    sphericity = ((36 * np.pi * volume**2)**(1/3)) / surface_area
    return min(sphericity, 1.0)  # Normalize to maximum of 1

def analyze_individual_pores(data, sample_name, output_dir='results'):
    """
    Analyze individual pores and save metrics to Excel and NPY files.
    """
    # Create binary mask
    binary_data = (data > 0)
    
    # Label connected components (pores)
    labeled_pores, num_pores = ndi.label(binary_data)
    
    # Save labeled pores array
    npy_filename = os.path.join(output_dir, f"{sample_name}_labeled_pores.npy")
    np.save(npy_filename, labeled_pores)
    
    # Compute local thickness
    local_thickness = ps.filters.local_thickness(binary_data, mode='dt')
    
    # Initialize list to store metrics for each pore
    pore_metrics = []
    
    # Analyze each pore individually
    for pore_id in range(1, num_pores + 1):
        # Create mask for current pore
        pore_mask = (labeled_pores == pore_id)
        
        # Get local thickness values for this pore
        pore_lt = local_thickness[pore_mask]
        pore_lt_nonzero = pore_lt[pore_lt > 0]
        
        # Calculate surface area
        surface_area = calculate_surface_area(pore_mask)
        volume = np.sum(pore_mask)
        
        # Calculate metrics for this pore
        pore_data = {
            'Pore_ID': pore_id,
            'Volume': volume,
            'Max_Local_Thickness': np.max(pore_lt_nonzero) if pore_lt_nonzero.size > 0 else 0,
            'Min_Local_Thickness': np.min(pore_lt_nonzero) if pore_lt_nonzero.size > 0 else 0,
            'Mean_Local_Thickness': np.mean(pore_lt_nonzero) if pore_lt_nonzero.size > 0 else 0,
            'Std_Local_Thickness': np.std(pore_lt_nonzero) if pore_lt_nonzero.size > 0 else 0,
            'Surface_Area': surface_area,
            'Sphericity': calculate_sphericity(volume, surface_area),
            'Cylindricity': calculate_cylindricity(pore_mask),
            'Center_X': np.mean(np.where(pore_mask)[0]),
            'Center_Y': np.mean(np.where(pore_mask)[1]),
            'Center_Z': np.mean(np.where(pore_mask)[2])
        }
        
        pore_metrics.append(pore_data)
    
    return pd.DataFrame(pore_metrics)

def process_mapped_clusters(input_dir, output_dir):
    """
    Process all samples in the mapped_clusters directory structure.

    Parameters:
    -----------
    input_dir : str
        Path to the mapped_clusters directory
    output_dir : str
        Path to save the analysis results
    """
    os.makedirs(output_dir, exist_ok=True)
    sample_folders = [f for f in os.listdir(input_dir) 
                      if os.path.isdir(os.path.join(input_dir, f))]

    all_results = {}

    for sample_name in sample_folders:
        if sample_name == "pore_analysis_results":
            print(f"Skipping folder: {sample_name}")
            continue

        print(f"Processing sample: {sample_name}")
        sample_output_dir = os.path.join(output_dir, sample_name)
        excel_path = os.path.join(sample_output_dir, f"{sample_name}_pore_metrics.xlsx")

        # Check if output and Excel already exist
        if os.path.exists(sample_output_dir) and os.path.exists(excel_path):
            print(f"  Results already exist for {sample_name}, reading from Excel...")
            combined_df = pd.read_excel(excel_path, sheet_name='All_Clusters')
            all_results[sample_name] = combined_df
            continue

        os.makedirs(sample_output_dir, exist_ok=True)

        sample_dir = os.path.join(input_dir, sample_name)
        cluster_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.npy')])

        sample_results = {}

        for cluster_file in cluster_files:
            print(f"  Processing cluster: {cluster_file}")
            cluster_path = os.path.join(sample_dir, cluster_file)
            cluster_data = np.load(cluster_path)
            cluster_num = int(cluster_file.split('_cluster_')[1].split('.')[0])

            df = analyze_individual_pores(
                cluster_data,
                f"{sample_name}_cluster_{cluster_num}",
                sample_output_dir
            )

            df['Cluster'] = cluster_num
            sample_results[f"cluster_{cluster_num}"] = df

        combined_df = pd.concat(sample_results.values(), ignore_index=True)
        all_results[sample_name] = combined_df

        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            combined_df.to_excel(writer, sheet_name='All_Clusters', index=False)
            for cluster_name, cluster_df in sample_results.items():
                cluster_df.to_excel(writer, sheet_name=cluster_name, index=False)

    summary_path = os.path.join(output_dir, 'all_samples_summary.xlsx')
    with pd.ExcelWriter(summary_path, engine='openpyxl') as writer:
        for sample_name, df in all_results.items():
            df.to_excel(writer, sheet_name=sample_name[:31], index=False)

        summary_stats = []
        for sample_name, df in all_results.items():
            for cluster in range(1, 6):
                cluster_data = df[df['Cluster'] == cluster]
                if not cluster_data.empty:
                    stats = {
                        'Sample': sample_name,
                        'Cluster': cluster,
                        'Total_Pores': len(cluster_data),
                        'Total_Volume': cluster_data['Volume'].sum(),
                        'Mean_Volume': cluster_data['Volume'].mean(),
                        'Mean_Sphericity': cluster_data['Sphericity'].mean(),
                        'Mean_Cylindricity': cluster_data['Cylindricity'].mean(),
                        'Mean_Local_Thickness': cluster_data['Mean_Local_Thickness'].mean()
                    }
                    summary_stats.append(stats)

        summary_df = pd.DataFrame(summary_stats)
        summary_df.to_excel(writer, sheet_name='Summary_Statistics', index=False)

    print(f"Analysis complete. Results saved in {output_dir}")

In [ ]:
if __name__ == "__main__":
    # Set your paths here
    INPUT_DIR = "../Prediction/ClusterInfo/mapped_clusters"
    OUTPUT_DIR = "../Prediction/ClusterInfo/mapped_clusters/pore_analysis_results"
    
    # Run the analysis
    process_mapped_clusters(INPUT_DIR, OUTPUT_DIR) 

Change the samples in ClusterInfo with RefinedClusterInfo:

In [14]:
if __name__ == "__main__":
    # Set your paths here
    INPUT_DIR = "../Prediction/RefinedClusterInfo/mapped_clusters"
    OUTPUT_DIR = "../Prediction/RefinedClusterInfo/mapped_clusters/pore_analysis_results"
    
    # Run the analysis
    process_mapped_clusters(INPUT_DIR, OUTPUT_DIR) 

Processing sample: DV8-sp6-1993
  Processing cluster: DV8-sp6-1993_cluster_1.npy


  0%|          | 0/25 [00:00<?, ?it/s]

  Processing cluster: DV8-sp6-1993_cluster_2.npy


  0%|          | 0/25 [00:00<?, ?it/s]

  Processing cluster: DV8-sp6-1993_cluster_3.npy


...

  Processing cluster: 068_MOM_E_clavatum_210_07_cluster_4.npy


  0%|          | 0/25 [00:00<?, ?it/s]

  Processing cluster: 068_MOM_E_clavatum_210_07_cluster_5.npy


  0%|          | 0/25 [00:00<?, ?it/s]

Analysis complete. Results saved in ../Prediction/RefinedClusterInfo/mapped_clusters/pore_analysis_results


#### Our data

In [ ]:
from scipy.stats import kruskal
import scikit_posthocs as sp

# [FORAM ADAPTATION] Load from our CSV output instead of multi-sheet Excel
# Original: INPUT_EXCEL = "../Prediction/ClusterInfo/mapped_clusters/pore_analysis_results/all_samples_summary.xlsx"
#           excel_file = pd.ExcelFile(INPUT_EXCEL)
#           sheet_names = [sheet for sheet in excel_file.sheet_names if sheet != 'Summary_Statistics']
INPUT_CSV = os.path.join(OUTPUT_RESULTS, 'quantification_results.csv')
df_stats = pd.read_csv(INPUT_CSV)

# [FORAM ADAPTATION] Derive Foram and Chamber columns from File_Name
# Original assumed one sheet per sample with a 'Cluster' column
df_stats[['Foram', 'Chamber']] = df_stats['File_Name'].str.split('_class_', expand=True)
df_stats['Chamber'] = df_stats['Chamber'].astype(int)
sheet_names = df_stats['Foram'].unique()

# Initialize lists to store results
results_list = []

# Process each sample (foram)
for sheet_name in sheet_names:
    df = df_stats[df_stats['Foram'] == sheet_name].copy()

    # [FORAM ADAPTATION] Metrics to test; original used ['Volume','Surface_Area','Mean_Local_Thickness']
    metrics = ['Pore_Total_Volume', 'Pore_Mean_Volume', 'Pore_Mean_LT']

    for metric in metrics:
        # [FORAM ADAPTATION] Groups from Chamber column (dynamic); original: range(1,6)
        chambers = sorted(df['Chamber'].unique())
        cluster_data = [df[df['Chamber'] == i][metric].values for i in chambers]

        # Remove empty clusters
        cluster_data = [data for data in cluster_data if len(data) > 0]
        if len(cluster_data) < 2:
            continue

        # Perform Kruskal-Wallis test
        h_stat, p_val = kruskal(*cluster_data)

        # Initialize dunn test results string
        dunn_results_str = ''

        # If significant, perform Dunn's test
        if p_val < 0.05:
            dunn_df = pd.DataFrame({
                'Value': np.concatenate(cluster_data),
                'Group': np.concatenate([[f'Ch{c}'] * len(data)
                                         for c, data in zip(chambers, cluster_data)])
            })

            # Perform Dunn's test
            dunn_results = sp.posthoc_dunn(dunn_df, val_col='Value', group_col='Group',
                                           p_adjust='bonferroni')

            # Convert Dunn's results to string format
            significant_pairs = []
            for i in range(len(dunn_results)):
                for j in range(i + 1, len(dunn_results)):
                    if dunn_results.iloc[i, j] < 0.05:
                        significant_pairs.append(
                            f'{dunn_results.index[i]}-{dunn_results.columns[j]}')
            dunn_results_str = '; '.join(significant_pairs)

        # Store results
        results_list.append({
            'Sample': sheet_name,
            'Metric': metric,
            'Kruskal_H': h_stat,
            'P_value': p_val,
            'Significant_Cluster_Pairs': dunn_results_str
        })

# Create results DataFrame and save
results_df = pd.DataFrame(results_list)
# [FORAM ADAPTATION] Save as CSV instead of Excel
# Original: results_df.to_excel(os.path.join(os.path.dirname(INPUT_EXCEL), 'statistical_analysis_results.xlsx'), index=False)
results_df.to_csv(os.path.join(OUTPUT_RESULTS, 'statistical_analysis_results.csv'), index=False)

In [4]:
from scipy.stats import wilcoxon
from scipy.stats import binomtest

# Define paths to Excel files and filter criteria
CLUSTER_EXCEL = "../Prediction/ClusterInfo/mapped_clusters/pore_analysis_results/all_samples_summary.xlsx"
FORAMS_EXCEL = "../Prediction/February/results/quantification_results.xlsx"

# Read data from both Excel files
# Read all sheets from cluster Excel file
excel_file = pd.ExcelFile(CLUSTER_EXCEL)
all_sheets = excel_file.sheet_names
cluster_data = {}

# Load each sample's detailed data
for sheet in all_sheets:
    if sheet != 'Summary_Statistics':  # Skip the summary sheet
        cluster_data[sheet] = pd.read_excel(CLUSTER_EXCEL, sheet_name=sheet)

# Read forams data and get the mean values
forams_df = pd.read_excel(FORAMS_EXCEL)
foram_mean_volume = forams_df['Pore_Mean_Volume'].mean()
foram_mean_lt = forams_df['Pore_Mean_LT'].mean()

# Initialize results list
comparison_results = []

# Compare each sample's cluster 2 with foram means
for sample, df in cluster_data.items():
    # Get cluster 2 data for this sample
    cluster2_data = df[df['Cluster'] == 2]

    # Remove rows where max local thickness is 1
    cluster2_data = cluster2_data[cluster2_data['Max_Local_Thickness'] > 1]
    
    if len(cluster2_data) > 0:  # Only process if cluster 2 exists
        # Wilcoxon signed-rank test to compare cluster 2 volumes against foram mean
        # This is non-parametric and doesn't assume normality
        volume_differences = cluster2_data['Volume'] - foram_mean_volume
        volume_stat, volume_p = wilcoxon(volume_differences,
                                       alternative='two-sided')
        
        # Calculate effect size using rank-biserial correlation
        # This is more appropriate for non-parametric tests
        n = len(volume_differences)
        r = 1 - (2 * volume_stat) / (n * (n + 1))
        
        # Additional descriptive statistics
        mean_diff = cluster2_data['Volume'].mean() - foram_mean_volume
        percent_diff = (mean_diff / foram_mean_volume) * 100
        
        # Wilcoxon test for local thickness comparison
        lt_differences = cluster2_data['Mean_Local_Thickness'] - foram_mean_lt
        lt_stat, lt_p = wilcoxon(lt_differences, alternative='two-sided')
        
        # Binomial test checks if the proportion of values above the mean
        # is significantly different from 0.5 (chance level)
        # k = number of successes (values above mean)
        # n = total number of trials
        # p = probability of success on each trial (0.5 = equal chance)
        # Returns pvalue to test if distribution differs from random chance
        volume_binom_p = binomtest(sum(cluster2_data['Volume'] > foram_mean_volume),
                                  n=len(cluster2_data), 
                                  p=0.5,
                                  alternative='two-sided')
        
        lt_binom_p = binomtest(sum(cluster2_data['Mean_Local_Thickness'] > foram_mean_lt),
                               n=len(cluster2_data),
                               p=0.5, 
                               alternative='two-sided')
        
        # Store results
        comparison_results.append({
            'Sample': sample,
            'Cluster2_Count': len(cluster2_data),
            'Volume_Wilcoxon_stat': volume_stat,
            'Volume_Wilcoxon_p': volume_p,
            'Volume_Wilcoxon_Significant': volume_p < 0.05,
            'Volume_Binom_p': volume_binom_p.pvalue,
            'Volume_Binom_Significant': volume_binom_p.pvalue < 0.05,
            'Volume_Above_Mean': sum(cluster2_data['Volume'] > foram_mean_volume),
            'LT_Wilcoxon_stat': lt_stat, 
            'LT_Wilcoxon_p': lt_p,
            'LT_Wilcoxon_Significant': lt_p < 0.05,
            'LT_Binom_p': lt_binom_p.pvalue,
            'LT_Binom_Significant': lt_binom_p.pvalue < 0.05,
            'LT_Above_Mean': sum(cluster2_data['Mean_Local_Thickness'] > foram_mean_lt),
            'Foram_Mean_Volume': foram_mean_volume,
            'Foram_Mean_LT': foram_mean_lt,
            'Cluster2_Mean_Volume': cluster2_data['Volume'].mean(),
            'Cluster2_Mean_LT': cluster2_data['Mean_Local_Thickness'].mean()
        })

# Create and save results DataFrame
results_df = pd.DataFrame(comparison_results)
results_df.to_excel('../Prediction/cluster2_forams_comparison.xlsx', index=False)

# Count significant differences
vol_wilcoxon_sig = sum(results_df['Volume_Wilcoxon_p'] < 0.05)
vol_binom_sig = sum(results_df['Volume_Binom_p'] < 0.05)
lt_wilcoxon_sig = sum(results_df['LT_Wilcoxon_p'] < 0.05)
lt_binom_sig = sum(results_df['LT_Binom_p'] < 0.05)
total_samples = len(results_df)

print(f"\nResults Summary:")
print(f"Out of {total_samples} samples:")
print(f"Volume comparisons:")
print(f"- {vol_wilcoxon_sig} samples show significant differences (Wilcoxon test)")
print(f"- {vol_binom_sig} samples show significant differences (Binomial test)")
print(f"\nLocal Thickness comparisons:") 
print(f"- {lt_wilcoxon_sig} samples show significant differences (Wilcoxon test)")
print(f"- {lt_binom_sig} samples show significant differences (Binomial test)")



Results Summary:
Out of 72 samples:
Volume comparisons:
- 68 samples show significant differences (Wilcoxon test)
- 65 samples show significant differences (Binomial test)

Local Thickness comparisons:
- 57 samples show significant differences (Wilcoxon test)
- 60 samples show significant differences (Binomial test)
